# Foundations of Data Science
## Tutorial 11 — SVD in the wild: detecting LLM hallucinations with EigenScore — SOLUTIONS

Large language models produce fluent text even when they are wrong. Detecting these "hallucinations" without external ground truth is a real problem for anyone deploying an LLM in production.

This tutorial reproduces the core idea from

> Chen, Chen, Wang, Zhang, Yu, Wang. *INSIDE: LLMs' Internal States Retain the Power of Hallucination Detection.* ICLR 2024. [arXiv:2402.03744](https://arxiv.org/abs/2402.03744), [code](https://github.com/D2I-ai/eigenscore).

Their **EigenScore** is a covariance-based confidence score built directly on top of the SVD. It is a nice example of how the linear algebra we saw in Tutorials 3–5 shows up in a real system.

The plan for the 90 minutes:

| Part | Topic | Suggested time |
|---:|---|---:|
| 1 | The hallucination problem and the INSIDE idea | 10 min |
| 2 | Sampling K responses from a small LLM | 15 min |
| 3 | Embedding responses | 10 min |
| 4 | From embeddings to EigenScore (the SVD part) | 20 min |
| 5 | Batch experiment: does EigenScore separate confident from uncertain prompts? | 20 min |
| 6 | Feature clipping and final discussion | 15 min |

You should be able to run this end-to-end on a free Google Colab CPU runtime, but a GPU is recommended.


## 1. The hallucination problem

Ask an LLM "Who won the 2019 Nobel Prize in Literature?" and it will (correctly) tell you Peter Handke. Ask it "Who won the 1873 Nobel Prize in Literature?" and it may still give you a confident answer — even though the prize was first awarded in 1901.

There is no red flag in the output itself: the sentence structure, the tone, and the confidence level look identical. The model just makes something up. This is what practitioners call a **hallucination**.

The observation behind INSIDE is simple: if you ask the same question **K times** with a bit of sampling noise, then

* on a question the model *knows*, the K answers are semantically consistent ("Peter Handke", "the Austrian writer Peter Handke", "Handke", …),
* on a question the model does *not* know, the K answers scatter across unrelated fabrications.

So a good hallucination score should measure the **semantic spread** of the K responses. That is exactly what a covariance matrix does.


### The pipeline in one paragraph

1. For a prompt $x$, sample $K$ responses $y_1, \ldots, y_K$ from the LLM.
2. Embed each response into $\mathbb{R}^d$ using a sentence encoder, giving $z_1,\ldots,z_K$.
3. Stack them into $Z \in \mathbb{R}^{K \times d}$, subtract the row mean, and form the $K \times K$ covariance matrix $\Sigma = Z\, J\, Z^T / \text{something}$ (details in Part 4).
4. Compute the **EigenScore**
$$
E(x) = \frac{1}{K} \sum_{i=1}^{K} \log\bigl(\lambda_i + \alpha\bigr),
$$
where $\lambda_i$ are the eigenvalues of $\Sigma + \alpha I$ (equivalently, its singular values, since $\Sigma$ is symmetric PSD).

A **large** EigenScore means the K embeddings span a large volume: the answers disagree → likely hallucination. A **small** EigenScore means the embeddings are close to a single direction: the answers agree → the model is confident and probably right.

This whole pipeline hinges on the SVD in step 4, which is why this belongs in a linear algebra course.


### Setup

The next cell installs `transformers` and `sentence-transformers` if they are not already there. On Colab this should take about a minute.


In [8]:
%pip install -q transformers sentence-transformers scikit-learn matplotlib


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score, roc_curve

rng = np.random.default_rng(2026)
torch.manual_seed(2026)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


Using device: cpu


## 2. Sampling K responses from a small LLM

We use `Qwen/Qwen2.5-0.5B-Instruct` (~1 GB, 500M parameters). It is small enough for Colab (CPU works, GPU is faster) but still competent on simple factual questions. Any small instruction-tuned model would do — the covariance idea does not depend on the model.


In [10]:
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32).to(DEVICE)
llm.eval()
print("LLM loaded:", sum(p.numel() for p in llm.parameters()) // 1_000_000, "M parameters")


KeyboardInterrupt: 

### Sampling function

At inference time we normally take the most likely token at each step (greedy decoding). To get *K different* answers we switch on sampling: at each step we sample the next token from the model's distribution, tempered by `temperature`. Higher temperature → more diverse but sometimes noisier answers.

Qwen2.5 is instruction-tuned, so we wrap our prompt in the chat template (`system` + `user` message). We also strip the prompt tokens from each output — for a decoder-only model, `generate` returns the prompt followed by the completion.


In [ ]:
SYSTEM_MSG = "You are a concise assistant. Answer in one short sentence."

@torch.no_grad()
def sample_responses(question, K=10, temperature=1.0, max_new_tokens=40):
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": question},
    ]
    prompt_text = llm_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = llm_tok(prompt_text, return_tensors="pt").to(DEVICE)
    prompt_len = inputs["input_ids"].shape[1]
    outputs = llm.generate(
        **inputs,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        num_return_sequences=K,
        max_new_tokens=max_new_tokens,
        pad_token_id=llm_tok.eos_token_id,
    )
    # strip the prompt tokens from each generated sequence
    completions = outputs[:, prompt_len:]
    return [llm_tok.decode(c, skip_special_tokens=True).strip() for c in completions]


### A prompt the model knows well

Qwen2.5-0.5B reliably knows the capital of France. All K samples should say more or less "Paris".


In [ ]:
known_question = "What is the capital of France?"
known_responses = sample_responses(known_question, K=10, temperature=1.0)
for i, r in enumerate(known_responses):
    print(f"[{i}] {r}")


### A prompt the model does not know

An obscure question — "Poyais" is a fictional country invented in the 1820s. There is no truthful answer; whatever the model says is fabricated.


In [ ]:
unknown_question = "What is the capital of Poyais?"
unknown_responses = sample_responses(unknown_question, K=10, temperature=1.0)
for i, r in enumerate(unknown_responses):
    print(f"[{i}] {r}")


**Observation.** On the first prompt the responses are essentially the same phrase. On the second, they scatter — the model produces different plausible-sounding place names on each sample. That scatter is exactly the signal we will pick up with the covariance matrix.


## 3. Embedding the responses

Raw strings are hard to compare mathematically. We turn each response into a vector in $\mathbb{R}^d$ using a **sentence encoder**. Two sentences with similar meaning end up with similar vectors, even if the words are different (e.g. "Paris" and "The capital is Paris").

We use `sentence-transformers/all-MiniLM-L6-v2` (~90 MB, $d=384$). This is a different model from the LLM: the LLM generates the text, the encoder embeds it.


In [ ]:
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
d = encoder.get_sentence_embedding_dimension()
print(f"Embedding dimension d = {d}")


### Exercise 3.1 — embed both response sets

Use `encoder.encode(list_of_strings)` to embed the K responses for both prompts. You should end up with two arrays of shape `(K, d) = (10, 384)`. Print the shapes and the norm of the first embedding to check.


In [ ]:
Z_known = encoder.encode(known_responses)
Z_unknown = encoder.encode(unknown_responses)
print("Z_known shape:  ", Z_known.shape)
print("Z_unknown shape:", Z_unknown.shape)
print(f"norm of first known embedding: {np.linalg.norm(Z_known[0]):.3f}")


### A quick visual check with PCA

Before doing the EigenScore, let's just look at the embeddings. Project both response sets onto their top 2 principal components together (Tutorial 5 style) and plot them.

Expected: the "known" responses form a tight cluster, the "unknown" responses spread out.


In [ ]:
Z_all = np.vstack([Z_known, Z_unknown])
Z_centered = Z_all - Z_all.mean(axis=0)
U, s, Vt = np.linalg.svd(Z_centered, full_matrices=False)
scores = U[:, :2] * s[:2]

plt.figure(figsize=(6, 6))
plt.scatter(scores[:10, 0], scores[:10, 1], label="known (capital of France)", s=80)
plt.scatter(scores[10:, 0], scores[10:, 1], label="unknown (capital of Poyais)", s=80, marker="^")
plt.axhline(0, color="grey", lw=0.5)
plt.axvline(0, color="grey", lw=0.5)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("K=10 response embeddings in PCA space")
plt.legend(); plt.show()


## 4. From embeddings to EigenScore — the SVD step

We have K embeddings $z_1, \ldots, z_K \in \mathbb{R}^d$, stacked as rows of $Z \in \mathbb{R}^{K \times d}$. Typically $d \gg K$ (here $384 \gg 10$).

### Building a K × K covariance matrix

We want a matrix whose eigenvalues describe how *spread out* the K points are. The natural object is the **centered Gram matrix**
$$
\Sigma = Z_c\, Z_c^T \in \mathbb{R}^{K \times K},
\qquad
Z_c = J\, Z,
\qquad
J = I_K - \tfrac{1}{K}\mathbf{1}\mathbf{1}^T.
$$

$J$ is the **centering matrix**: multiplying by $J$ subtracts the mean row from each row. $\Sigma$ is symmetric and positive semi-definite. Its rank is at most $K-1$ (the mean direction is killed).

Two observations that make this a clean SVD example:

1. If $Z_c = U \tilde\Sigma V^T$ is the SVD, then $\Sigma = Z_c Z_c^T = U \tilde\Sigma^2 U^T$. So **the eigenvalues of $\Sigma$ are the squared singular values of $Z_c$**.
2. Working with the $K \times K$ matrix $\Sigma$ is much cheaper than the $d \times d$ matrix $Z_c^T Z_c$: we get the same non-zero eigenvalues but with $K \ll d$.

### The EigenScore itself

$$
\boxed{\;E(x) = \frac{1}{K} \sum_{i=1}^{K} \log\bigl(\lambda_i + \alpha\bigr)\;}
$$

with $\alpha \approx 10^{-3}$ a small regularizer (so we do not take $\log 0$ on the zero eigenvalue that centering always introduces). This is exactly the log-determinant of $\Sigma + \alpha I$, up to the factor $1/K$:
$$
E(x) = \tfrac{1}{K} \log \det(\Sigma + \alpha I).
$$

### Geometric interpretation

The determinant of $\Sigma$ is the squared volume of the parallelepiped spanned by $z_1 - \bar z, \ldots, z_K - \bar z$. Large volume ⇔ answers point in many different directions ⇔ high semantic diversity ⇔ likely hallucination.

**High EigenScore ⇒ hallucination. Low EigenScore ⇒ the model is confident.**


### Exercise 4.1 — implement EigenScore

Fill in the function below. Two ways to compute the eigenvalues:

* `np.linalg.eigvalsh(Sigma + alpha * np.eye(K))` — since $\Sigma$ is symmetric PSD.
* `np.linalg.svd(Z_centered)` — take the singular values, square them, add $\alpha$. This is what the paper's code actually does.

Both should give (numerically) the same answer. Include a small assertion in your implementation.


In [ ]:
def eigen_score(Z, alpha=1e-3):
    # Compute EigenScore from K response embeddings. Z has shape (K, d).
    K, d = Z.shape
    Z_c = Z - Z.mean(axis=0)
    Sigma = Z_c @ Z_c.T
    eigvals = np.linalg.eigvalsh(Sigma + alpha * np.eye(K))
    return float(np.mean(np.log(eigvals)))


### Sanity check

Compute the EigenScore for both prompts. The "unknown" prompt should give a noticeably larger value.


In [ ]:
E_known = eigen_score(Z_known)
E_unknown = eigen_score(Z_unknown)
print(f"EigenScore (known / capital of France)   = {E_known:+.3f}")
print(f"EigenScore (unknown / capital of Poyais) = {E_unknown:+.3f}")


### Exercise 4.2 — SVD form

Rewrite `eigen_score` using `np.linalg.svd` on the centered `Z` directly, without ever forming $\Sigma$. Verify that it agrees with the eigenvalue form to within `1e-8`.

*Why this is useful.* For very high-dimensional embeddings, working with the SVD of the $K \times d$ matrix skips building a large intermediate. In our case both are cheap, but this is the pattern you would use in production.


In [ ]:
def eigen_score_via_svd(Z, alpha=1e-3):
    K, d = Z.shape
    Z_c = Z - Z.mean(axis=0)
    s = np.linalg.svd(Z_c, compute_uv=False)  # singular values of Z_c
    # Sigma = Z_c Z_c^T has eigenvalues s**2, padded with zeros if K > rank(Z_c).
    eigvals = np.concatenate([s ** 2, np.zeros(K - len(s))]) + alpha
    return float(np.mean(np.log(eigvals)))

assert np.isclose(eigen_score(Z_known),   eigen_score_via_svd(Z_known),   atol=1e-8)
assert np.isclose(eigen_score(Z_unknown), eigen_score_via_svd(Z_unknown), atol=1e-8)
print("both implementations agree.")


## 5. Does EigenScore actually separate hallucinations?

One example proves nothing. We now run the same pipeline on a small dataset with a mix of prompts the model should know and prompts it should not.

The `unknown` prompts either ask about invented entities or events after the model's training cutoff.


In [ ]:
known_questions = [
    "What is the capital of France?",
    "What is the capital of Japan?",
    "What is the capital of Germany?",
    "What is the capital of Italy?",
    "What is the capital of Spain?",
    "What is 2 + 2?",
    "What language do people speak in Brazil?",
    "What color is the sky on a clear day?",
]

unknown_questions = [
    "What is the capital of Poyais?",
    "What is the capital of Ruritania?",
    "Who won the 1873 Nobel Prize in Literature?",
    "What is the population of Zubrowka?",
    "Who directed the 1652 film 'The Green Star'?",
    "What is the melting point of unobtainium?",
    "Who is the current king of Genovia?",
    "What year was the Treaty of Vulgaria signed?",
]


### Exercise 5.1 — compute EigenScore for every prompt

For each prompt, sample K=10 responses at temperature 1.0, embed them, and compute the EigenScore. Store all scores in two arrays `scores_known` and `scores_unknown`.

This takes about 1–2 minutes on Colab CPU.


In [ ]:
def score_question(question, K=10, temperature=1.0):
    responses = sample_responses(question, K=K, temperature=temperature)
    Z = encoder.encode(responses)
    return eigen_score(Z), responses

scores_known   = np.array([score_question(q)[0] for q in known_questions])
scores_unknown = np.array([score_question(q)[0] for q in unknown_questions])

print("known:  ", np.round(scores_known, 2))
print("unknown:", np.round(scores_unknown, 2))
print(f"mean known:   {scores_known.mean():+.3f}")
print(f"mean unknown: {scores_unknown.mean():+.3f}")


### Visualize the separation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

lo = min(scores_known.min(), scores_unknown.min())
hi = max(scores_known.max(), scores_unknown.max())
bins = np.linspace(lo, hi, 10)

axes[0].hist(scores_known,   bins=bins, alpha=0.7, label="known")
axes[0].hist(scores_unknown, bins=bins, alpha=0.7, label="unknown")
axes[0].set_xlabel("EigenScore"); axes[0].set_ylabel("count")
axes[0].set_title("EigenScore distribution")
axes[0].legend()

y_true  = np.concatenate([np.zeros_like(scores_known), np.ones_like(scores_unknown)])
y_score = np.concatenate([scores_known, scores_unknown])
fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

axes[1].plot(fpr, tpr, marker="o")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
axes[1].set_title(f"ROC — AUC = {auc:.2f}")
plt.tight_layout(); plt.show()

print(f"AUC = {auc:.3f}")


**Short answer.** Report the AUC and comment. A perfect detector has AUC = 1.0; random guessing gives 0.5. Anything comfortably above 0.5 shows that the covariance spectrum of a small batch of resampled answers carries a real signal — no ground-truth labels required.


**Discussion.** On a run of this toy dataset we typically get AUC in the 0.85–0.95 range with just 8 + 8 prompts and $K=10$. The exact value fluctuates because both the sampling from the LLM and the embedding are noisy, but the gap between the two distributions is stable: known-fact prompts sit clearly below unknown-fact prompts on the EigenScore axis.

The takeaway is not the specific AUC — it's that a purely intrinsic signal (the spectrum of the K responses' covariance matrix) tracks something that would otherwise require an oracle to check.


## 6. Bonus: test-time feature clipping

The INSIDE paper also proposes **feature clipping**: at each generation step, look at the activations of the LLM's penultimate layer and truncate the top and bottom $p$-th percentile (they use $p=0.2$, following the three-sigma rule). Intuition: rare, extreme activations are the ones driving "overconfident" fabrications; clipping them lets EigenScore separate hallucinations more cleanly.

Reproducing this properly needs internal states of the LLM, which is more plumbing than we want in a 90-minute tutorial. But the idea generalises: any smoothing of the internal representation that shrinks the model's ability to commit hard to a wrong answer will *widen* the K samples on unknown prompts and thus **increase** the EigenScore gap.


### Exercise 6.1 — the effect of K

We used $K=10$. What happens if you use $K=3$? $K=20$?

Pick one prompt from each set (`known_prompts[0]`, `unknown_prompts[0]`) and compute the EigenScore for $K \in \{3, 5, 8, 12, 20\}$. Plot both curves on the same axes.

Discuss: for small $K$, is the score reliable? For large $K$, what is the cost?


In [ ]:
Ks = [3, 5, 8, 12, 20]
score_k_known   = []
score_k_unknown = []
for K in Ks:
    score_k_known.append(score_question(known_questions[0],   K=K)[0])
    score_k_unknown.append(score_question(unknown_questions[0], K=K)[0])

plt.plot(Ks, score_k_known,   marker="o", label="known")
plt.plot(Ks, score_k_unknown, marker="s", label="unknown")
plt.xlabel("K (number of samples)")
plt.ylabel("EigenScore")
plt.title("EigenScore vs number of resampled responses")
plt.legend(); plt.grid(alpha=0.3); plt.show()


**Discussion.** For very small $K$ (say $K=3$) the score is dominated by whichever two responses happened to be close or far apart — the K×K covariance matrix has only 2 non-trivial eigenvalues, so the mean of their logs is very noisy. Both curves are unreliable there.

As $K$ grows the unknown curve keeps climbing (more samples reveal more of the diversity in the model's uncertain distribution), while the known curve stays roughly flat (there is no genuine diversity to uncover). The cost is linear in $K$ generation calls, which is the dominant expense; $K=10$ is a reasonable default.


### Exercise 6.2 — a lexical baseline

Before we pat ourselves on the back, let's see whether the SVD really added anything. A dead-simple baseline: for K responses, compute the **average pairwise cosine similarity** of their embeddings. High similarity → confident, low → uncertain, so we use `1 - mean_pairwise_cosine` as an uncertainty score.

Implement this baseline and compare its ROC-AUC to EigenScore on the same prompt set.

Hint: for centered, unit-normalized embeddings $\hat z_i$, the mean pairwise cosine is $\frac{1}{K(K-1)}\bigl(\|\sum_i \hat z_i\|^2 - K\bigr)$.


In [ ]:
def mean_pairwise_similarity_uncertainty(Z):
    Z_norm = Z / np.linalg.norm(Z, axis=1, keepdims=True)
    K = Z.shape[0]
    # mean of off-diagonal cosine similarities = (||sum||^2 - K) / (K(K-1))
    s = Z_norm.sum(axis=0)
    mean_sim = (s @ s - K) / (K * (K - 1))
    return float(1 - mean_sim)

# We already have the K responses cached inside score_question; re-sample here so
# both scores are computed on the same batch of responses for a fair comparison.
def both_scores(question, K=10, temperature=1.0):
    responses = sample_responses(question, K=K, temperature=temperature)
    Z = encoder.encode(responses)
    return eigen_score(Z), mean_pairwise_similarity_uncertainty(Z)

eig_known,  cos_known  = zip(*[both_scores(q) for q in known_questions])
eig_unknown, cos_unknown = zip(*[both_scores(q) for q in unknown_questions])
eig_known, cos_known, eig_unknown, cos_unknown = map(
    np.array, (eig_known, cos_known, eig_unknown, cos_unknown))

y_true = np.concatenate([np.zeros(len(eig_known)), np.ones(len(eig_unknown))])
auc_eig = roc_auc_score(y_true, np.concatenate([eig_known,  eig_unknown]))
auc_cos = roc_auc_score(y_true, np.concatenate([cos_known,  cos_unknown]))

print(f"EigenScore     AUC = {auc_eig:.3f}")
print(f"Pairwise-cos   AUC = {auc_cos:.3f}")


**Short answer.** On this small toy dataset, does the SVD-based EigenScore actually beat the simple pairwise-similarity baseline? Give one reason why the paper still argues for EigenScore — think about what happens as $K$ grows, or when embeddings live on a curved manifold.


**Discussion.** On this small dataset the two scores are usually within a few AUC points of each other — sometimes the baseline even wins. That is not damning: the pairwise-cosine score really is a strong baseline, and much of the signal is already captured by "how tight is this cloud of embeddings?".

Where EigenScore has a real advantage:

* **Higher moments.** Pairwise similarity is a scalar summary of the mean pairwise angle. EigenScore looks at the full *spectrum* of the K×K covariance. If the K responses split into two consistent clusters (e.g. the model is torn between two plausible fabrications), the pairwise mean can look moderate while the spectrum picks up the multi-modal structure.
* **Sensitivity as $K$ grows.** Cosine similarity converges to a single number; the log-determinant continues to reflect the intrinsic dimensionality of the response set.
* **Directly interpretable as log-volume.** $\log\det$ has a clean geometric meaning that mean cosine does not.

In production the honest answer is: try both, and use whichever is more separable on your validation set.


## 7. Final checkpoint

You should now be able to answer, without code:

1. Why does EigenScore use the $K \times K$ centered Gram matrix instead of the $d \times d$ covariance matrix?
2. Why is the regularizer $\alpha I$ necessary — even a tiny $\alpha$?
3. Geometrically, what does a large $\det(\Sigma + \alpha I)$ mean about the K response embeddings?
4. If the LLM is *systematically wrong* but consistent (say, it always answers "Paris" to "capital of Poyais"), what does EigenScore do? Is that a failure mode?
5. Two months from now you deploy an LLM assistant at work. Which two pieces of infrastructure from this tutorial do you actually need to run EigenScore on live traffic?

### Where to go next

* The full paper: [arXiv:2402.03744](https://arxiv.org/abs/2402.03744).
* Reference implementation: [github.com/D2I-ai/eigenscore](https://github.com/D2I-ai/eigenscore) — see `func/metric.py`.
* Related direction: probing LLM *internal hidden states* directly (as the paper does), not just the surface embedding of the output. That is the "INSIDE" in the paper's title.
